In [1]:
import sys
import os

def get_UGCE_directory():
    """Get the path of the 'UGCE-User-Guided-Counterfactual-Exploration' directory."""
    current_dir = os.getcwd()
    target_dir = 'UGCE-User-Guided-Counterfactual-Exploration'
    
    while os.path.basename(current_dir) != target_dir:
        current_dir = os.path.dirname(current_dir)
        if current_dir == os.path.dirname(current_dir):
            return None
        
    return current_dir

def get_system_slash():
    """Get the system-specific directory separator."""
    return os.sep

UGCE_dir = get_UGCE_directory()
sys.path.append(UGCE_dir)
sep = get_system_slash()
sys.path.append(UGCE_dir + get_system_slash() + 'src')

from dataLoader import *
from utils import *
from test_utils import *

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
seed_number = 42
import random

random.seed(seed_number)
np.random.seed(seed_number)

In [4]:
datasetName = 'AdultCA'

In [5]:
from folktables import ACSDataSource, ACSIncome

# Initialize the data source for California in 2024
data_source = ACSDataSource(survey_year='2023', horizon='1-Year', survey='person')

# Download and extract the data for California
ca_data = data_source.get_data(states=['CA'], download=True)

# Define the ACSIncome task
features, label, group = ACSIncome.df_to_numpy(ca_data)
feature_names = ACSIncome.features

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import dice_ml
from dice_ml.utils import helpers

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

dataset = pd.DataFrame(features, columns=feature_names)
TARGET_COLUMN = 'target'
dataset['target'] = label
dataset['target'] = LabelEncoder().fit_transform(dataset['target'])
target = dataset['target']
datasetX = dataset.drop('target', axis=1)

x_train, x_test, y_train, y_test = train_test_split(datasetX,
                                                    target,
                                                    test_size=0.2,
                                                    random_state=0,
                                                    stratify=target)

numerical = datasetX.columns.to_list()
categorical = x_train.columns.difference(numerical)

try:
    import joblib
    model = joblib.load(f"{ugce_dir}/results/models/{datasetName}_model.pkl")
except:
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))])

    transformations = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical),
            ('cat', categorical_transformer, categorical)])

    model = RandomForestClassifier(random_state=42)

    model = Pipeline(steps=[('preprocessor', transformations),
                        ('classifier', model)])

    model.fit(x_train, y_train)

    import joblib
    os.makedirs(f"{ugce_dir}/results/models", exist_ok=True)
    joblib.dump(model, f"{ugce_dir}/results/models/{datasetName}_model.pkl")

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

negative_instances = x_test[model.predict(x_test) == 0]
instances_to_explain = negative_instances

Accuracy:  0.8063754427390791


# UGCE

## Dynamic

# Only Immutability

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    'AGEP': 'i'
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints

results_incremental_explainer_immutability = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_immutability.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_immutability, open(f'{results_dir}/results_incremental{strategy}_only_immutability_consts_ONE_constraint.pkl', 'wb'))

In [9]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer_immutability, verbose=True)

Time taken for generating counterfactuals using UGCE dynamic:  47.640332674980165  minutes
Full Time: mean = 47.64, std = 0.00
Generations: mean = 6.00, std = 0.00
Coverage: mean = 97.95, std = 0.00
Proximity Loss: mean = 0.14, std = 0.00
Sparsity: mean = 0.06, std = 0.00
Intermediate Best Distances: mean = 0.06, std = 0.00


(47.640332674980165,
 6.001869536658077,
 97.95100222717149,
 0.13508039220592147,
 0.05608298072887078,
 0.06395986897961006)

# Only Range

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    'AGEP': (42, 94)
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints

results_incremental_explainer_ranges = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_ranges.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_ranges, open(f'{results_dir}/results_incremental{strategy}_only_range_consts_ONE_constraint.pkl', 'wb'))

In [ ]:
from test_utils import *

aggregate_results_incremental(iea, results_incremental_explainer_ranges, verbose=True)

Time taken for generating counterfactuals using UGCE dynamic:  21.191643341382346  minutes
Full Time: mean = 21.19, std = 0.00
Generations: mean = 6.00, std = 0.00
Coverage: mean = 46.24, std = 0.00
Proximity Loss: mean = 0.11, std = 0.00
Sparsity: mean = 0.07, std = 0.00
Intermediate Best Distances: mean = 0.02, std = 0.00


(21.191643341382346,
 6.0,
 46.23608017817372,
 0.10744052364912333,
 0.06513264138326491,
 0.020535182020123277)

# Only Directionality

In [ ]:
from ugce import *

iea = UGCE(model, datasetX,
           numerical, categorical)

updated_constraints = {
    'AGEP': 'incr'
}
for col in iea.feature_names:
    if col not in updated_constraints:
        updated_constraints[col] = ''
updated_constraints


results_incremental_explainer_direct = []
for i in range(5):
    import time
    strategy = "fix_population_update_fitness"
    results_incremental = iea.explain_instances(instances_to_explain, seed_number=None,
        dynamic_constraints=True, constraints={},
        initial_population_variability=0.7, data_distribution=True,
        strategy="fix_population_update_fitness", population_size_dynamic=0, cfes_requested=2,
        num_generations=500, initial_population_strategy="kdtree", population_size=10, early_stopping_iterations=5,
        num_parents=19, selection_method="sus", tournsize=5, crossover_points=2,
        elite_ratio=0.1, cxpb=0.8, mutpb=0.7, regeneration_tries_intermediate=0,
        regeneration_tries_intermediate_after_updating_constraints=0,
        updated_constraints=updated_constraints, automatic_user_acceptance=True,
        verbose=False, running_times_per_instance=1,
        distance_metric="weighted_l1", multistep_updated_constraints=False,
        normalized_fitness=True, reweighting_lambdas_after_generations=-1,
        initial_lambdas_without_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        initial_lambdas_with_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_after_updating_constraints={"lambda1":0.2, "lambda2":0.2, "lambda3":1},
        lambdas_reweighting_after_updating_constraints_and_some_generations={"lambda1":0.2, "lambda2":0.2, "lambda3":1})
    results_incremental_explainer_direct.append(results_incremental)
import os
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
os.makedirs(results_dir, exist_ok=True)
pickle.dump(results_incremental_explainer_direct, open(f'{results_dir}/results_incremental{strategy}_only_directionality_consts_ONE_constraint.pkl', 'wb'))

In [ ]:
import pickle
results_dir = f'{UGCE_dir}/results/assess_ugce/{datasetName}/incremental'
results_incremental_explainer_direct = pickle.load(open(f'{results_dir}/results_incremental{strategy}_only_directionality_consts_ONE_constraint.pkl', 'rb'))
from test_utils import *
aggregate_results_incremental(iea, results_incremental_explainer_direct, verbose=True)

Time taken for generating counterfactuals using UGCE dynamic:  17.840735320250193  minutes
Full Time: mean = 17.84, std = 0.00
Generations: mean = 6.00, std = 0.00
Coverage: mean = 34.03, std = 0.00
Proximity Loss: mean = 0.11, std = 0.00
Sparsity: mean = 0.07, std = 0.00
Intermediate Best Distances: mean = 0.02, std = 0.00


(17.840735320250193,
 6.0,
 34.026231130908194,
 0.11097261321235088,
 0.0664960718294042,
 0.016650395943256024)